# 03 - Fine-Tuning Experiments
## Multilingual Health QA - MLT1 Final Project

This notebook contains Experiments 2 onward: fine-tuning `mT5-small` using LoRA (PEFT) for parameter-efficient training, followed by hyperparameter and configuration variations.

**Approach:** We use LoRA to keep memory usage low on the T4 GPU while still adapting the full model's behavior for the QA task.

In [ ]:
!pip install -q -U transformers
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, MT5ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

DATA_DIR = '/kaggle/input/datasets/jokjohnkur/health-qa-data-v3'

train = pd.read_csv(f'{DATA_DIR}/Train.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')

# Drop the 1 row with empty input (found during EDA)
train = train[train['input'].str.strip() != ''].reset_index(drop=True)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
from transformers import MT5ForConditionalGeneration

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name).to(device)

print("Model loaded successfully:", model_name)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Data Preprocessing

We split the data into train/validation (90/10, stratified by language), then define a tokenization function that:
- Adds a task prefix prompt to the input
- Tokenizes both input and target text
- Sets max lengths based on EDA findings (questions ~15 words avg, answers ~76 words avg, Akan up to 458 words)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train, test_size=0.1, random_state=42, stratify=train['subset']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))

# Preprocessing: add task prefix
PREFIX = "answer health question: "
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 256

def preprocess(examples):
    inputs = [PREFIX + str(x) for x in examples['input']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = tokenizer(text_target=examples['output'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print("Preprocessing function defined")
print("Prefix:", repr(PREFIX))
print("Max input length:", MAX_INPUT_LEN)
print("Max target length:", MAX_TARGET_LEN)

#### **Convert to HF Dataset format and tokenize**

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[['input', 'output', 'subset']])
val_ds = Dataset.from_pandas(val_df[['input', 'output', 'subset']])

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])
val_tokenized = val_ds.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])

print("Train tokenized:", train_tokenized)
print()
print("Sample input_ids length:", len(train_tokenized[0]['input_ids']))
print("Sample labels length:", len(train_tokenized[0]['labels']))

## LoRA Configuration

LoRA (Low-Rank Adaptation) freezes the base model weights and injects small trainable rank-decomposition matrices into the attention layers. This drastically reduces trainable parameters (and memory usage), making fine-tuning feasible on a T4 GPU.

**Config for Experiment 2:** r=16, targeting query/value projection matrices (`q`, `v`) - standard starting point for T5-family models.

In [ ]:
!pip install -q peft

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training Setup

Using Hugging Face `Seq2SeqTrainer` with:
- Batch size 8 (fits comfortably on T4 with mT5-small + LoRA)
- 2 epochs (initial run - we'll assess if more epochs help in later experiments)
- Mixed precision (fp16) for speed
- Learning rate 1e-3 (LoRA typically uses higher LR than full fine-tuning)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-lora-exp2",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=2,
    fp16=True,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Trainer configured")
print("Steps per epoch:", len(train_tokenized) // training_args.per_device_train_batch_size)
print("Total steps:", (len(train_tokenized) // training_args.per_device_train_batch_size) * training_args.num_train_epochs)

In [ ]:
trainer.train()

## Experiment 2 Evaluation

Generate predictions on the same 240-example validation sample used for the baseline, and compute ROUGE-1/ROUGE-L for direct comparison with Experiment 1.

In [ ]:
model.eval()

# Reuse the same sampling approach as Experiment 1 for fair comparison
val_sample2 = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)

predictions2 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions2.append(pred)
    if idx < 5:
        print(f"[{row['subset']}] Pred: {pred[:150]}")
        print(f"   Ref:  {row['output'][:150]}")
        print()

val_sample2['prediction'] = predictions2
print("Done generating predictions.")

#### **computing ROUGE on these predictions**

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

In [ ]:
rouge1_scores2 = []
rougeL_scores2 = []

for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction'])
    rouge1_scores2.append(scores['rouge1'].fmeasure)
    rougeL_scores2.append(scores['rougeL'].fmeasure)

val_sample2['rouge1'] = rouge1_scores2
val_sample2['rougeL'] = rougeL_scores2

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores2):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores2):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1','rougeL']].mean().round(4))

**Result (validation sample, n=240):**
- ROUGE-1 F1: 0.0922 (37x improvement over baseline)
- ROUGE-L F1: 0.0833 (40x improvement over baseline)
- Best languages: English-Ghana (0.164), Akan (0.155)
- Worst language: Amharic (0.0019) - likely due to smallest training set (1,845 samples) + distinct Ge'ez script

**Insight:** Fine-tuning with LoRA produces a massive jump from zero-shot, confirming the model CAN learn this task. However, the repetition-loop problem (noted above) likely caps the score - fixing decoding parameters (repetition_penalty, no_repeat_ngram_size) without retraining could give a "free" improvement. This becomes Experiment 3.

## Experiment 3: Improved Decoding (Repetition Penalty)

**Hypothesis:** The repetition loops observed in Experiment 2 are a decoding artifact, not purely a training issue. Adding `repetition_penalty` and `no_repeat_ngram_size` should produce cleaner, more diverse text from the SAME trained model - a free improvement without retraining.

**What changed:** Generation parameters only (repetition_penalty=1.3, no_repeat_ngram_size=3). Model weights unchanged from Experiment 2.

In [ ]:
predictions3 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions3.append(pred)
    if idx < 5:
        print(f"[{row['subset']}] Pred: {pred[:150]}")

val_sample2['prediction3'] = predictions3
print("Done.")

In [ ]:
rouge1_scores3 = []
rougeL_scores3 = []

for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction3'])
    rouge1_scores3.append(scores['rouge1'].fmeasure)
    rougeL_scores3.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v3'] = rouge1_scores3
val_sample2['rougeL_v3'] = rougeL_scores3

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores3):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores3):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1_v3','rougeL_v3']].mean().round(4))
print()
print("=== Comparison: Exp2 vs Exp3 ===")
print(f"Exp2 ROUGE-1: {np.mean(rouge1_scores2):.4f}  →  Exp3 ROUGE-1: {np.mean(rouge1_scores3):.4f}")
print(f"Exp2 ROUGE-L: {np.mean(rougeL_scores2):.4f}  →  Exp3 ROUGE-L: {np.mean(rougeL_scores3):.4f}")

**Result (validation sample, n=240):**
- ROUGE-1 F1: 0.1520 (+65% relative improvement over Exp2)
- ROUGE-L F1: 0.1228 (+47% relative improvement over Exp2)
- Best languages: English-Ethiopia (0.263), Akan (0.234), Kiswahili (0.180)
- Amharic improved slightly (0.0019 → 0.0133) but remains the weakest

**Insight:** Repetition penalty and n-gram blocking were the dominant factor limiting Experiment 2's score — the underlying fine-tuned weights were better than they appeared. This is a low-cost, high-impact change (zero retraining required). Amharic remains the bottleneck, confirming the data scarcity issue identified in EDA - this motivates Experiment 4 (oversampling or weighted loss for Amharic).

In [ ]:
test_predictions2 = []

for idx, row in test.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    test_predictions2.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)}")

print("Done!")

In [ ]:
submission2 = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_predictions2,
    'TargetR1F1': test_predictions2,
    'TargetLLM': test_predictions2,
})

submission2.to_csv('submission_exp3_lora_reppenalty.csv', index=False)
print("Saved!")
submission2.head()

**Zindi Public Leaderboard Score: 0.145043** (Submission #2, screenshot saved)
**Progression: 0.000676 → 0.145043 (214x improvement from Experiment 1)**

## Experiment 4: Oversample Amharic in Training Data

**Hypothesis:** Amharic's poor performance (ROUGE-1: 0.0133 vs overall 0.1520) is primarily due to having the fewest training examples (1,845 vs 7,624 for Eng_Uga). Oversampling Amharic examples should improve its representation during training without hurting other languages much.

**What changed:** Duplicate Amharic training examples 3x before tokenization. Retrain from scratch with same LoRA config (r=16, 2 epochs).

In [ ]:
# Oversample Amharic examples 3x
amh_rows = train_df[train_df['subset'] == 'Amh_Eth']
train_df_oversampled = pd.concat([train_df] + [amh_rows]*2, ignore_index=True)
train_df_oversampled = train_df_oversampled.sample(frac=1, random_state=42).reset_index(drop=True)

print("Original train size:", len(train_df))
print("Oversampled train size:", len(train_df_oversampled))
print()
print("New subset distribution:")
print(train_df_oversampled['subset'].value_counts())

In [ ]:
# Reload fresh base model + new LoRA adapters
model_fresh = MT5ForConditionalGeneration.from_pretrained("google/mt5-small").to(device)
model_fresh = get_peft_model(model_fresh, lora_config)
model_fresh.print_trainable_parameters()

# Tokenize oversampled training set
train_ds_v2 = Dataset.from_pandas(train_df_oversampled[['input', 'output', 'subset']])
train_tokenized_v2 = train_ds_v2.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])

print("Tokenized oversampled train:", train_tokenized_v2)

#### **Set up trainer and train Experiment 4**

In [ ]:
training_args_v2 = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-lora-exp4",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=2,
    fp16=True,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

data_collator_v2 = DataCollatorForSeq2Seq(tokenizer, model=model_fresh, padding=True)

trainer_v2 = Seq2SeqTrainer(
    model=model_fresh,
    args=training_args_v2,
    train_dataset=train_tokenized_v2,
    eval_dataset=val_tokenized,
    data_collator=data_collator_v2,
)

trainer_v2.train()

## Experiment 4 Evaluation

Using the SAME validation sample and SAME decoding settings (repetition_penalty=1.3, no_repeat_ngram_size=3) as Experiment 3, for a fair comparison.

In [ ]:
model_fresh.eval()

predictions4 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_fresh.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions4.append(pred)

val_sample2['prediction4'] = predictions4

rouge1_scores4 = []
rougeL_scores4 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction4'])
    rouge1_scores4.append(scores['rouge1'].fmeasure)
    rougeL_scores4.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v4'] = rouge1_scores4
val_sample2['rougeL_v4'] = rougeL_scores4

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores4):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores4):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1_v4','rougeL_v4']].mean().round(4))
print()
print("=== Comparison: Exp3 vs Exp4 ===")
print(f"Exp3 ROUGE-1: {np.mean(rouge1_scores3):.4f}  →  Exp4 ROUGE-1: {np.mean(rouge1_scores4):.4f}")
print(f"Exp3 ROUGE-L: {np.mean(rougeL_scores3):.4f}  →  Exp4 ROUGE-L: {np.mean(rougeL_scores4):.4f}")
print()
print("Amharic specifically:")
print(f"Exp3 Amharic ROUGE-1: {val_sample2[val_sample2['subset']=='Amh_Eth']['rouge1_v3'].mean():.4f}")
print(f"Exp4 Amharic ROUGE-1: {val_sample2[val_sample2['subset']=='Amh_Eth']['rouge1_v4'].mean():.4f}")

**Result (validation sample, n=240):**
- ROUGE-1 F1: 0.1548 (vs Exp3: 0.1520) - marginal overall improvement
- ROUGE-L F1: 0.1286 (vs Exp3: 0.1228)
- Amharic ROUGE-1: 0.0133 - IDENTICAL to Exp3, no improvement at all

**Insight:** The hypothesis was NOT supported - oversampling Amharic 3x did not improve Amharic-specific performance whatsoever (exact same score). The slight overall improvement is more likely attributable to the larger total dataset size (30,152 vs 26,832 examples) providing more training steps overall, not targeted Amharic learning. This suggests Amharic's poor performance isn't primarily a sample-count problem - it may instead be a **tokenization/vocabulary coverage issue**: mT5's SentencePiece vocabulary may poorly represent Ge'ez script, fragmenting Amharic words into many subword tokens and making generation harder regardless of how much Amharic data is seen. This is worth investigating directly (e.g., checking average token count per Amharic word) as a future improvement direction.

## **Setup**

In [14]:
#Consolidated Setup (re-run after session restart)
!pip install -q -U transformers

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, MT5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from sklearn.model_selection import train_test_split
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from rouge_score import rouge_scorer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("GPU count:", torch.cuda.device_count())  # should print 1

# Load data
DATA_DIR = '/kaggle/input/datasets/jokjohnkur/health-qa-data-v3'
train = pd.read_csv(f'{DATA_DIR}/Train.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')
train = train[train['input'].str.strip() != ''].reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

train_df, val_df = train_test_split(train, test_size=0.1, random_state=42, stratify=train['subset'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

PREFIX = "answer health question: "
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 256

def preprocess(examples):
    inputs = [PREFIX + str(x) for x in examples['input']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = tokenizer(text_target=examples['output'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q", "v"],
)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

val_sample2 = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42), include_groups=False
).reset_index(drop=True)

print("Setup complete")
print("Train:", len(train_df), "| Val:", len(val_df), "| Val sample:", len(val_sample2))

Device: cuda
GPU count: 2
Setup complete
Train: 26832 | Val: 2982 | Val sample: 240


## Experiment 5: Investigating Amharic's Persistent Low Score

**Hypothesis:** Amharic's flat 0.0133 ROUGE score (unchanged by oversampling) suggests a tokenization issue rather than a data-quantity issue. mT5's SentencePiece vocabulary may fragment Amharic (Ge'ez script) into excessive subword tokens, making generation structurally harder.

**What we're checking:** Average tokens-per-word ratio across languages - a high ratio for Amharic would support this hypothesis.

In [ ]:
# Compare tokenization efficiency across languages
results = {}
for lang in train['subset'].unique():
    sample_texts = train[train['subset']==lang]['output'].head(50)
    total_words = 0
    total_tokens = 0
    for text in sample_texts:
        words = len(text.split())
        tokens = len(tokenizer.encode(text))
        total_words += words
        total_tokens += tokens
    results[lang] = total_tokens / total_words if total_words > 0 else 0

for lang, ratio in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{lang}: {ratio:.2f} tokens/word")

**Result:**

| Language | Tokens/Word |
|---|---|
| Amharic | 3.52 |
| Luganda | 3.01 |
| Akan | 2.38 |
| Kiswahili | 2.00 |
| English (all variants) | 1.58-1.69 |

**Insight:** Amharic and Luganda require roughly 2x more subword tokens per word than English to represent the same content. This confirms mT5's SentencePiece vocabulary has poor coverage for Ge'ez script (Amharic) and Bantu morphology (Luganda) – generating a 20-word Amharic answer actually requires the model to correctly predict ~70 tokens in sequence, compounding error rates token-by-token. This is a fundamental limitation of using a general multilingual model rather than an Africa-specific one (e.g., AfriMT5), and explains why oversampling alone (Experiment 4) couldn't fix it – more examples of the same hard-to-tokenise text doesn't reduce the token-level difficulty. A genuinely Africa-focused pretrained model would be the proper fix but is out of scope given time constraints – noted as a limitation/future improvement in the report.

## Experiment 6: Extended Training (5 Epochs)

**Hypothesis:** Validation loss was still decreasing at epoch 2 (3.98→3.73 in Exp2). More epochs should continue improving generation quality before overfitting sets in.

**What changed:** num_train_epochs: 2 → 5. Same LoRA config (r=16), same data (non-oversampled, since Exp4 showed oversampling doesn't help).

In [ ]:
# Tokenize datasets (needed again after session restart)
train_ds = Dataset.from_pandas(train_df[['input', 'output', 'subset']])
val_ds = Dataset.from_pandas(val_df[['input', 'output', 'subset']])

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])
val_tokenized = val_ds.map(preprocess, batched=True, remove_columns=['input', 'output', 'subset'])

# Fresh model + LoRA
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small").to(device)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("Ready for Experiment 6")

In [ ]:
training_args_v3 = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-lora-exp6",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=5,
    fp16=True,
    logging_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

data_collator_v3 = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer_v3 = Seq2SeqTrainer(
    model=model,
    args=training_args_v3,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator_v3,
)

trainer_v3.train()

#### ***Evaluating Experiment 6 on validation sample***

In [ ]:
model.eval()

predictions6 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions6.append(pred)

val_sample2['prediction6'] = predictions6

rouge1_scores6 = []
rougeL_scores6 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction6'])
    rouge1_scores6.append(scores['rouge1'].fmeasure)
    rougeL_scores6.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v6'] = rouge1_scores6
val_sample2['rougeL_v6'] = rougeL_scores6

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores6):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores6):.4f}")

In [ ]:
# Recover the 'subset' column (deterministic sampling, same order)
val_sample2_subsets = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)

# Verify alignment before merging
assert val_sample2['input'].equals(val_sample2_subsets['input']), "Order mismatch!"

val_sample2['subset'] = val_sample2_subsets['subset'].values
print("Subset column recovered")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1_v6','rougeL_v6']].mean().round(4))

## Experiment 7: mT5-Base + Expanded LoRA + Adafactor Optimizer

**Hypothesis:** Given the leaderboard gap (0.145 vs 0.56 for competitive scores), mT5-small's capacity (300M params) may be the binding constraint. mT5-base (580M params) with a higher LoRA rank (32 vs 16) and the Adafactor optimizer (more memory-efficient, allowing the larger model to fit) should provide a substantial capability increase.

**What changed:**
- Model: mT5-small → mT5-base
- LoRA rank: 16 → 32
- Optimizer: AdamW → Adafactor
- save_total_limit: 1 → 3 (enables checkpoint averaging in Experiment 9)
- Epochs: 3 (balance between Exp2's 2 and Exp6's 5, given larger model = more time per epoch)

In [ ]:
from transformers import AutoTokenizer, MT5ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# Load mT5-base
tokenizer_base = AutoTokenizer.from_pretrained("google/mt5-base")
model_base = MT5ForConditionalGeneration.from_pretrained("google/mt5-base").to(device)

# Re-tokenize with mT5-base tokenizer (same SentencePiece vocab as mT5-small, but verify)
def preprocess_base(examples):
    inputs = [PREFIX + str(x) for x in examples['input']]
    model_inputs = tokenizer_base(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = tokenizer_base(text_target=examples['output'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_pandas(train_df[['input', 'output', 'subset']])
val_ds = Dataset.from_pandas(val_df[['input', 'output', 'subset']])

train_tokenized_base = train_ds.map(preprocess_base, batched=True, remove_columns=['input', 'output', 'subset'])
val_tokenized_base = val_ds.map(preprocess_base, batched=True, remove_columns=['input', 'output', 'subset'])

# LoRA config: rank 32, expanded targets
lora_config_v2 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q", "v"],
)

model_base = get_peft_model(model_base, lora_config_v2)
model_base.print_trainable_parameters()

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args_v4 = Seq2SeqTrainingArguments(
    output_dir="./mt5-base-lora-exp7",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-3,
    num_train_epochs=1,
    fp16=True,
    optim="adafactor",
    logging_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)
data_collator_v4 = DataCollatorForSeq2Seq(tokenizer_base, model=model_base, padding=True)
trainer_v4 = Seq2SeqTrainer(
    model=model_base,
    args=training_args_v4,
    train_dataset=train_tokenized_base,
    eval_dataset=val_tokenized_base,
    data_collator=data_collator_v4,
)
trainer_v4.train()

**Note:** Due to severe time constraints on the T4 GPU (3 epochs estimated at 4.9 hours), this experiment was scaled to 1 epoch (1.6 hours) – still a meaningful single-epoch comparison against mT5-small's first-epoch results from Experiment 2 (val loss 3.98 after epoch 1).

In [ ]:
# Recover subset column (same fix as before)
val_sample2_subsets = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)
val_sample2['subset'] = val_sample2_subsets['subset'].values

model_base.eval()

predictions7 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_base(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_base.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
    predictions7.append(pred)
    if idx < 5:
        print(f"[{row['subset']}] Pred: {pred[:150]}")

val_sample2['prediction7'] = predictions7

rouge1_scores7 = []
rougeL_scores7 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction7'])
    rouge1_scores7.append(scores['rouge1'].fmeasure)
    rougeL_scores7.append(scores['rougeL'].fmeasure)

val_sample2['rouge1_v7'] = rouge1_scores7
val_sample2['rougeL_v7'] = rougeL_scores7

print(f"\nOverall ROUGE-1 F1: {np.mean(rouge1_scores7):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores7):.4f}")
print()
print("Per-language breakdown:")
print(val_sample2.groupby('subset')[['rouge1_v7','rougeL_v7']].mean().round(4))
print()
print("=== Comparison ===")
print(f"Exp3 (mT5-small, 2ep): ROUGE-1=0.1520, ROUGE-L=0.1228, Zindi=0.145043")
print(f"Exp6 (mT5-small, 5ep): ROUGE-1=0.1444, ROUGE-L=0.1193")
print(f"Exp7 (mT5-base, 1ep):  ROUGE-1={np.mean(rouge1_scores7):.4f}, ROUGE-L={np.mean(rougeL_scores7):.4f}")

**Result (validation sample, n=240):**
- Overall ROUGE-1 F1: 0.1928 (vs Exp3 best: 0.1520, +27%)
- Overall ROUGE-L F1: 0.1565 (vs Exp3 best: 0.1228, +27%)
- Amharic ROUGE-1: 0.0667 (vs 0.0133 in all mT5-small experiments) - 5x improvement
- Improved across 7/8 languages vs Exp3; only Eng_Eth/Lug_Uga roughly flat
- Training anomaly: average training_loss=19.27 vs final validation_loss=2.976, suggesting early-training instability with Adafactor+fp16+LoRA r=32 on a freshly-initialized larger model, which stabilized by epoch end

**Insight:** This is the strongest evidence yet that mT5-small's capacity (300M params) was the primary bottleneck, particularly for Amharic - confirming Experiment 5's tokenization-fragmentation hypothesis (Amharic needs ~3.5 tokens/word; a larger model's increased representational capacity per token appears to partially compensate for this). Despite only 1 epoch (vs 2-5 for mT5-small experiments), mT5-base achieves the best results across nearly all languages, demonstrating that model selection had more impact than any data-centric or hyperparameter-centric intervention (Experiments 4, 6) attempted on mT5-small.

In [ ]:
test_predictions7 = []

for idx, row in test.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_base(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_base.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
    test_predictions7.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)}")

print("Done!")

In [ ]:
submission7 = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_predictions7,
    'TargetR1F1': test_predictions7,
    'TargetLLM': test_predictions7,
})

submission7.to_csv('submission_exp7_mt5base.csv', index=False)
print("Saved!")
submission7.head()

**Zindi Public Leaderboard Score: 0.211963** (Submission #3, rank 465/543)
**Progression: 0.000676 → 0.145043 → 0.211963 (313x improvement from baseline)**

## Experiment 8: Greedy vs Beam Search Decoding

**Hypothesis:** Beam search (num_beams=4) explores multiple hypotheses and typically improves fluency/ROUGE-L over greedy decoding, but at higher inference cost. Testing greedy decoding on the mT5-base model quantifies this tradeoff.

**What changed:** Generation parameter only - `num_beams=1` (greedy) vs `num_beams=4` (beam search). Same model (Experiment 7's mT5-base), same repetition_penalty/no_repeat_ngram_size.

In [ ]:
import time
start = time.time()

predictions8 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_base(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_base.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=1,  # greedy
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )
    pred = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
    predictions8.append(pred)

val_sample2['prediction8'] = predictions8

rouge1_scores8 = []
rougeL_scores8 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction8'])
    rouge1_scores8.append(scores['rouge1'].fmeasure)
    rougeL_scores8.append(scores['rougeL'].fmeasure)

print(f"Greedy decoding time: {(time.time()-start)/60:.1f} min")
print(f"Greedy ROUGE-1: {np.mean(rouge1_scores8):.4f}  (Beam ROUGE-1 was: {np.mean(rouge1_scores7):.4f})")
print(f"Greedy ROUGE-L: {np.mean(rougeL_scores8):.4f}  (Beam ROUGE-L was: {np.mean(rougeL_scores7):.4f})")

**Result (validation sample, n=240):**
- Greedy: ROUGE-1=0.1998, ROUGE-L=0.1493, time=4.6 min
- Beam (num_beams=4): ROUGE-1=0.1928, ROUGE-L=0.1565, time=~12 min

**Insight:** Decoding strategy produces metric-specific tradeoffs. Greedy decoding improved ROUGE-1 (unigram/lexical overlap) but decreased ROUGE-L (longest common subsequence, capturing fluency/structure) - likely because greedy selects high-probability individual words matching common vocabulary, while beam search's multi-hypothesis exploration produces more structurally coherent sequences. Weighting both metrics equally (as the competition does, 0.37 each), the two approaches are nearly equivalent (greedy: 0.37×0.1998+0.37×0.1493=0.1292; beam: 0.37×0.1928+0.37×0.1565=0.1292) - but greedy achieves this at roughly 1/3 the inference cost, an important practical consideration for a system intended for real-world deployment (e.g., low-latency health worker assistant tools).

## Experiment 9: Per-Language Max Target Length

**Hypothesis:** Akan has the longest reference answers (avg 105.6 words, max 458) - our global `max_target_length=256` may truncate optimal Akan generations. Increasing it specifically for Akan should improve Akan's ROUGE without affecting other languages.

**What changed:** Inference-only - Akan examples generated with `max_length=384` instead of 256. Same model (Exp7), same beam search settings.

In [ ]:
import time
start = time.time()

aka_mask = val_sample2['subset'] == 'Aka_Gha'
predictions9 = val_sample2['prediction7'].copy()  # start from Exp7's beam predictions

for idx, row in val_sample2[aka_mask].iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_base(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_base.generate(
            **inputs,
            max_length=384,  # increased from 256
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
    predictions9[idx] = pred

val_sample2['prediction9'] = predictions9

rouge1_scores9 = []
rougeL_scores9 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction9'])
    rouge1_scores9.append(scores['rouge1'].fmeasure)
    rougeL_scores9.append(scores['rougeL'].fmeasure)

print(f"Time: {(time.time()-start)/60:.1f} min")
print(f"Overall ROUGE-1: {np.mean(rouge1_scores9):.4f}  (was {np.mean(rouge1_scores7):.4f})")
print(f"Overall ROUGE-L: {np.mean(rougeL_scores9):.4f}  (was {np.mean(rougeL_scores7):.4f})")
print()
print("Akan specifically:")
print(f"ROUGE-1: {np.mean(np.array(rouge1_scores9)[aka_mask.values]):.4f}  (was {np.mean(np.array(rouge1_scores7)[aka_mask.values]):.4f})")
print(f"ROUGE-L: {np.mean(np.array(rougeL_scores9)[aka_mask.values]):.4f}  (was {np.mean(np.array(rougeL_scores7)[aka_mask.values]):.4f})")

**Result (validation sample, n=240):**
- Overall ROUGE-1: 0.1928 (unchanged), ROUGE-L: 0.1565 (unchanged)
- Akan ROUGE-1: 0.3079 (was 0.3078), ROUGE-L: 0.2274 (was 0.2273) - negligible difference

**Insight:** The hypothesis was incorrect. Increasing max_target_length from 256→384 had no effect, indicating the model is NOT being truncated at 256 tokens for Akan – combined with `early_stopping=True` and `repetition_penalty`, the model already terminates generation (via EOS token) well before the length limit on its own. The bottleneck for Akan's score is generation QUALITY (word choice, coherence relative to reference), not generation LENGTH. This rules out a simple inference-time fix and suggests further Akan improvement would require additional training (more epochs or larger model), which is outside our remaining time budget.

## Experiment 10: Deterministic (Beam) vs Sampling-Based Decoding

**Hypothesis:** For sensitive health content, deterministic decoding (beam search) should produce more consistent, lower-variance outputs than sampling-based decoding (temperature=0.7), which introduces randomness that could increase hallucination risk - an important consideration for responsible deployment.

**What changed:** `do_sample=True, temperature=0.7` vs Experiment 7's deterministic beam search. Same model.

In [ ]:
import time
start = time.time()

predictions10 = []
for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_base(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_base.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )
    pred = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
    predictions10.append(pred)

val_sample2['prediction10'] = predictions10

rouge1_scores10 = []
rougeL_scores10 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction10'])
    rouge1_scores10.append(scores['rouge1'].fmeasure)
    rougeL_scores10.append(scores['rougeL'].fmeasure)

print(f"Time: {(time.time()-start)/60:.1f} min")
print(f"Sampling (T=0.7): ROUGE-1={np.mean(rouge1_scores10):.4f}, ROUGE-L={np.mean(rougeL_scores10):.4f}")
print(f"Beam (Exp7):      ROUGE-1={np.mean(rouge1_scores7):.4f}, ROUGE-L={np.mean(rougeL_scores7):.4f}")

**Result (validation sample, n=240):**
- Sampling (T=0.7, top_p=0.9): ROUGE-1=0.2110, ROUGE-L=0.1475
- Beam search (Exp7): ROUGE-1=0.1928, ROUGE-L=0.1565

**Insight:** Sampling produced a higher ROUGE-1 but lower ROUGE-L than beam search – consistent with Experiment 8's finding that less-constrained decoding favours lexical overlap (ROUGE-1) at the expense of sequence coherence (ROUGE-L). However, the more important consideration for this domain is **reproducibility and consistency**: beam search is deterministic – running it twice on the same input produces identical output – while sampling introduces randomness, meaning the same health question could receive different answers on different occasions. For a system providing maternal/reproductive health guidance, this non-determinism is a meaningful risk: inconsistent answers to the same question could undermine user trust or, in edge cases, produce a less accurate response by chance. Despite the marginal ROUGE-1 gain, **beam search (deterministic decoding) is the more responsible choice for deployment**, and is retained as the final configuration (consistent with Experiment 7's submission).

# Phase 2: Per-Language Specialization with AfriMT5

The 10 required experiments (above) established a working pipeline (mT5-base + LoRA, leaderboard score 0.211963) and identified two key bottlenecks: (1) general mT5's tokeniser poorly represents Amharic/Akan/Luganda (Experiment 5), and (2) a single shared adapter forces languages to compete for capacity (Experiment 6).

This phase tests two targeted fixes:
1. **AfriMT5** (`masakhane/afri-mt5-base`) - continued-pretrained on 17 African languages, potentially solving the tokenization fragmentation issue at its root
2. **Per-language LoRA adapters** - since the test set's `subset` column identifies each example's language, separate specialized adapters avoid cross-lingual interference entirely

## Step 1: AfriMT5 Tokenization Check

In [6]:
from transformers import AutoTokenizer, MT5ForConditionalGeneration

# Test AfriMT5 tokenizer
tokenizer_afri = AutoTokenizer.from_pretrained("masakhane/afri-mt5-base")

# Re-run Experiment 5's tokenization efficiency check
print("=== AfriMT5 tokenization efficiency ===")
results_afri = {}
for lang in train['subset'].unique():
    sample_texts = train[train['subset']==lang]['output'].head(50)
    total_words = 0
    total_tokens = 0
    for text in sample_texts:
        words = len(text.split())
        tokens = len(tokenizer_afri.encode(text))
        total_words += words
        total_tokens += tokens
    results_afri[lang] = total_tokens / total_words if total_words > 0 else 0

print("\nComparison: mT5 vs AfriMT5 (tokens/word)")
mt5_results = {'Amh_Eth': 3.52, 'Lug_Uga': 3.01, 'Aka_Gha': 2.38, 'Swa_Ken': 2.00, 
                'Eng_Gha': 1.69, 'Eng_Eth': 1.68, 'Eng_Uga': 1.67, 'Eng_Ken': 1.58}

for lang, ratio in sorted(results_afri.items(), key=lambda x: -mt5_results.get(x[0], 0)):
    print(f"{lang}: mT5={mt5_results.get(lang, 0):.2f}  →  AfriMT5={ratio:.2f}")

tokenizer_config.json:   0%|          | 0.00/408 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

=== AfriMT5 tokenization efficiency ===

Comparison: mT5 vs AfriMT5 (tokens/word)
Amh_Eth: mT5=3.52  →  AfriMT5=3.52
Lug_Uga: mT5=3.01  →  AfriMT5=3.01
Aka_Gha: mT5=2.38  →  AfriMT5=2.38
Swa_Ken: mT5=2.00  →  AfriMT5=2.00
Eng_Gha: mT5=1.69  →  AfriMT5=1.69
Eng_Eth: mT5=1.68  →  AfriMT5=1.68
Eng_Uga: mT5=1.67  →  AfriMT5=1.67
Eng_Ken: mT5=1.58  →  AfriMT5=1.58


#### ****Loading AfriMT5 model weights with Exp7's exact recipe****

In [7]:
from transformers import MT5ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

model_afri = MT5ForConditionalGeneration.from_pretrained("masakhane/afri-mt5-base").to(device)

lora_config_afri = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q", "v"],
)

model_afri = get_peft_model(model_afri, lora_config_afri)
model_afri.print_trainable_parameters()
print("AfriMT5 loaded with LoRA r=32")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 3,538,944 || all params: 585,940,224 || trainable%: 0.6040
AfriMT5 loaded with LoRA r=32


#### **Train AfriMT5 with Exp7's exact recipe (1 epoch)**

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

# Reload fresh AfriMT5 (current weights are corrupted from NaN training)
from peft import LoraConfig, get_peft_model, TaskType
from transformers import MT5ForConditionalGeneration

model_afri2 = MT5ForConditionalGeneration.from_pretrained("masakhane/afri-mt5-base").to(device)
lora_config_afri = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q", "v"],
)
model_afri2 = get_peft_model(model_afri2, lora_config_afri)

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args_afri2 = Seq2SeqTrainingArguments(
    output_dir="./afrimt5-lora-exp11-v2",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-3,
    num_train_epochs=1,
    fp16=False,  # disabled — caused NaN
    bf16=False,  # also disable for safety on T4 (T4 has limited bf16 support)
    optim="adafactor",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

data_collator_afri2 = DataCollatorForSeq2Seq(tokenizer_afri, model=model_afri2, padding=True)
trainer_afri2 = Seq2SeqTrainer(
    model=model_afri2,
    args=training_args_afri2,
    train_dataset=train_tokenized_base,
    eval_dataset=val_tokenized_base,
    data_collator=data_collator_afri2,
)
trainer_afri2.train()

In [ ]:
# Recover subset column
val_sample2_subsets = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)
val_sample2['subset'] = val_sample2_subsets['subset'].values

model_afri.eval()
predictions_afri = []

for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_afri(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_afri.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_afri.decode(outputs[0], skip_special_tokens=True)
    predictions_afri.append(pred)

val_sample2['prediction_afri'] = predictions_afri

rouge1_afri = []
rougeL_afri = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction_afri'])
    rouge1_afri.append(scores['rouge1'].fmeasure)
    rougeL_afri.append(scores['rougeL'].fmeasure)

print(f"AfriMT5 ROUGE-1: {np.mean(rouge1_afri):.4f}  (mT5-base Exp7: 0.1928)")
print(f"AfriMT5 ROUGE-L: {np.mean(rougeL_afri):.4f}  (mT5-base Exp7: 0.1565)")
print()
print("Per-language breakdown:")
lang_results = val_sample2.copy()
lang_results['r1'] = rouge1_afri
lang_results['rL'] = rougeL_afri
print(lang_results.groupby('subset')[['r1','rL']].mean().round(4))
print()
print("Amharic specifically:")
amh = lang_results[lang_results['subset']=='Amh_Eth']
print(f"ROUGE-1: {amh['r1'].mean():.4f}  (Exp7: 0.0667)")

In [ ]:
test_predictions_afri = []

for idx, row in test.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_afri(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_afri.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_afri.decode(outputs[0], skip_special_tokens=True)
    test_predictions_afri.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)}")

print("Done!")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from transformers import AutoTokenizer, MT5ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

tokenizer_afri = AutoTokenizer.from_pretrained("masakhane/afri-mt5-base")
model_afri = MT5ForConditionalGeneration.from_pretrained("masakhane/afri-mt5-base").to(device)

lora_config_afri = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q", "v"],
)
model_afri = get_peft_model(model_afri, lora_config_afri)
model_afri.print_trainable_parameters()
print("AfriMT5 ready")

In [8]:
train_ds = Dataset.from_pandas(train_df[['input', 'output', 'subset']])
val_ds = Dataset.from_pandas(val_df[['input', 'output', 'subset']])

def preprocess_base(examples):
    inputs = [PREFIX + str(x) for x in examples['input']]
    model_inputs = tokenizer_afri(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = tokenizer_afri(text_target=examples['output'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tokenized_base = train_ds.map(preprocess_base, batched=True, remove_columns=['input', 'output', 'subset'])
val_tokenized_base = val_ds.map(preprocess_base, batched=True, remove_columns=['input', 'output', 'subset'])
print("Datasets tokenized")

Map:   0%|          | 0/26832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2982 [00:00<?, ? examples/s]

Datasets tokenized


In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

# Recover subset column
val_sample2_subsets = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)
val_sample2['subset'] = val_sample2_subsets['subset'].values

model_afri2.eval()
predictions_afri2 = []

for idx, row in val_sample2.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_afri(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_afri2.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_afri.decode(outputs[0], skip_special_tokens=True)
    predictions_afri2.append(pred)

val_sample2['prediction_afri2'] = predictions_afri2

rouge1_afri2 = []
rougeL_afri2 = []
for idx, row in val_sample2.iterrows():
    scores = scorer.score(row['output'], row['prediction_afri2'])
    rouge1_afri2.append(scores['rouge1'].fmeasure)
    rougeL_afri2.append(scores['rougeL'].fmeasure)

print(f"AfriMT5 fp32 ROUGE-1: {np.mean(rouge1_afri2):.4f}")
print(f"AfriMT5 fp32 ROUGE-L: {np.mean(rougeL_afri2):.4f}")
print()
print("Per-language breakdown:")
lang_r = val_sample2.copy()
lang_r['r1'] = rouge1_afri2
lang_r['rL'] = rougeL_afri2
print(lang_r.groupby('subset')[['r1','rL']].mean().round(4))
print()
print("=== Full comparison ===")
print(f"Exp3  (mT5-small, 2ep, fp16):    ROUGE-1=0.1520, Zindi=0.145043")
print(f"Exp7  (mT5-base,  1ep, fp16):    ROUGE-1=0.1928, Zindi=0.211963")
print(f"AfriMT5 fp16 (broken):           ROUGE-1=N/A")
print(f"AfriMT5 fp32 (this):             ROUGE-1={np.mean(rouge1_afri2):.4f}")

In [ ]:
import time
start = time.time()

test_predictions_afri2 = []
for idx, row in test.iterrows():
    input_text = PREFIX + row['input']
    inputs = tokenizer_afri(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        outputs = model_afri2.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer_afri.decode(outputs[0], skip_special_tokens=True)
    test_predictions_afri2.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)} — {(time.time()-start)/60:.1f} min elapsed")

print(f"Done! Total time: {(time.time()-start)/60:.1f} minutes")

In [ ]:
submission_afri2 = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_predictions_afri2,
    'TargetR1F1': test_predictions_afri2,
    'TargetLLM': test_predictions_afri2,
})

submission_afri2.to_csv('submission_afri_mt5_fp32.csv', index=False)
print("Saved!")
print(f"Shape: {submission_afri2.shape}")
submission_afri2.head(3)

In [10]:
import os
import gc
import torch
torch.cuda.empty_cache()
gc.collect()
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


from transformers import MT5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# Define LoRA config
lora_config_afri = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q", "v"],
)

model_afri4 = MT5ForConditionalGeneration.from_pretrained("masakhane/afri-mt5-base").to(device)
model_afri4 = get_peft_model(model_afri4, lora_config_afri)
model_afri4.print_trainable_parameters()

training_args_afri4 = Seq2SeqTrainingArguments(
    output_dir="./afrimt5-lora-exp11-v4",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    num_train_epochs=2,
    fp16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

data_collator_afri4 = DataCollatorForSeq2Seq(tokenizer_afri, model=model_afri4, padding=True)
trainer_afri4 = Seq2SeqTrainer(
    model=model_afri4,
    args=training_args_afri4,
    train_dataset=train_tokenized_base,
    eval_dataset=val_tokenized_base,
    data_collator=data_collator_afri4,
)
trainer_afri4.train()

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 3,538,944 || all params: 585,940,224 || trainable%: 0.6040


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 